# L73 · 音乐嵌入向量 — 对比学习：相似风格靠近，不同流派拉远

**目标**：训练一个对比学习音乐编码器，让同歌距离近、异歌距离远，每首歌输出一个固定长度的语义向量。

🔗 **Aurora 连接**：本节实现 `aurora.music` 的核心 embedding 层——每首歌经 `MusicEncoder` 压缩为 128 维向量，后续检索、推荐、分类均以此向量为输入。

## 核心直觉

音乐相似性无法用标签距离衡量——两首爵士乐在风格上可能比一首爵士和一首摇滚更接近，但这种"接近"只存在于音频本身的结构里。对比学习的思路：给定同一首歌的两个片段作为正样本对，随机另一首歌的片段作为负样本，强迫编码器让正对靠近、负对拉开。训练完成后，向量空间的距离就自然编码了音乐语义相似度。

In [ ]:
import torch
import torch.nn as nn
import numpy as np

## 1. 对比学习：Triplet Loss

每次训练以三元组 (anchor, positive, negative) 为单位：
- `anchor`：参考片段
- `positive`：与 anchor 来自同一首歌的另一片段
- `negative`：来自不同歌的片段

**Triplet Loss 公式**：

```
L = mean( max(0,  d(anchor, pos) - d(anchor, neg) + margin) )
```

其中 `d` 是 L2 距离，`margin` 是一个安全边距（典型值 0.2）。当 `d_neg - d_pos > margin` 时 loss 为 0，说明负样本已足够远，不再产生梯度。

In [ ]:
# 直觉演示：loss 面
import torch

margin = 0.2
d_pos_vals = torch.linspace(0, 2, 50)
d_neg_fixed = 1.0  # 固定负样本距离

loss_vals = torch.clamp(d_pos_vals - d_neg_fixed + margin, min=0)

print("d_pos\t\td_neg\t\tloss")
for i in [0, 12, 25, 37, 49]:
    print(f"{d_pos_vals[i]:.2f}\t\t{d_neg_fixed:.2f}\t\t{loss_vals[i]:.4f}")

## 2. MusicEncoder：Mel → 固定长度向量

输入：mel 特征矩阵，形状 `(B, 1, T, n_mels)`，时间轴可变长。
架构：
```
Conv2d(1→16, k=3) → ReLU → MaxPool(2,2)
Conv2d(16→32, k=3) → ReLU → AdaptiveAvgPool → 全局压平
Linear(32*n_mels//4, embed_dim=128)
```
`AdaptiveAvgPool2d((1, n_mels//4))` 把时间轴压成 1，使输出长度与输入时长无关——这是处理可变长音频的关键。

In [ ]:
class MusicEncoder(nn.Module):
    def __init__(self, n_mels=64, embed_dim=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                       # T//2, n_mels//2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, n_mels // 4)),   # 1, n_mels//4
        )
        self.fc = nn.Linear(32 * (n_mels // 4), embed_dim)

    def forward(self, x):                             # x: (B,1,T,n_mels)
        h = self.cnn(x)                               # (B,32,1,n_mels//4)
        h = h.flatten(1)                              # (B, 32*n_mels//4)
        return self.fc(h)                             # (B, embed_dim)

# 验证输入输出形状
enc = MusicEncoder(n_mels=64, embed_dim=128)
dummy = torch.randn(4, 1, 100, 64)   # batch=4, T=100帧, n_mels=64
out = enc(dummy)
print(f"输入形状: {dummy.shape}  →  输出形状: {out.shape}")
assert out.shape == (4, 128), "形状不对"
print("✅ MusicEncoder 形状验证通过")

## 3. L2 归一化：让向量落在单位球面

训练后对输出做 L2 归一化：

```
v_norm = v / ||v||_2
```

归一化后向量模长 = 1，此时：
- **余弦相似度** = `v_a · v_b`（直接点积，无需除模长）
- L2 距离与余弦距离单调等价：`||a-b||^2 = 2 - 2*(a·b)`

这让 ANN 检索（FAISS 内积索引）可直接替代余弦搜索，速度更快。

In [ ]:
def l2_normalize(x: torch.Tensor) -> torch.Tensor:
    """对最后一维做 L2 归一化。"""
    return x / (x.norm(dim=-1, keepdim=True) + 1e-8)

v = torch.randn(3, 128)
v_n = l2_normalize(v)
norms = v_n.norm(dim=-1)
print(f"归一化后各向量的模长: {norms.tolist()}")
assert torch.allclose(norms, torch.ones(3), atol=1e-5)
print("✅ L2 归一化验证通过")

## 4. ✏️ 实现 `triplet_loss(anchor, positive, negative, margin=0.2)`

**推理路线**：
1. `d_pos = torch.norm(anchor - positive, dim=1)` — 每个样本的 L2 距离，shape (B,)
2. `d_neg = torch.norm(anchor - negative, dim=1)` — 同理
3. `raw = d_pos - d_neg + margin` — 若负样本足够远则 raw < 0
4. `loss = torch.clamp(raw, min=0).mean()` — 只对"还不够好"的三元组施加梯度

**参考输入输出**：
```
anchor == positive → d_pos = 0
negative = ones*10 → d_neg ≈ sqrt(d * 10^2) >> margin
→ raw = 0 - d_neg + 0.2 << 0 → loss = 0
```

<details><summary>点击查看参考实现</summary>

```python
def triplet_loss(anchor, positive, negative, margin=0.2):
    d_pos = torch.norm(anchor - positive, dim=1)
    d_neg = torch.norm(anchor - negative, dim=1)
    loss = torch.clamp(d_pos - d_neg + margin, min=0)
    return loss.mean()
```
</details>

In [ ]:
def triplet_loss(anchor, positive, negative, margin=0.2):
    """
    Triplet loss for contrastive learning.
    Args:
        anchor, positive, negative: torch.Tensor, shape (B, d)
        margin: float, safety margin
    Returns:
        scalar Tensor
    """
    # ✏️ TODO: 计算 d_pos = L2(anchor, positive) 每行距离
    d_pos = None

    # ✏️ TODO: 计算 d_neg = L2(anchor, negative) 每行距离
    d_neg = None

    # ✏️ TODO: loss = clamp(d_pos - d_neg + margin, min=0).mean()
    loss = None

    return loss

In [ ]:
# 检查1：anchor==positive, negative 很远 → loss ≈ 0
a = torch.zeros(4, 8)
p = torch.zeros(4, 8)
n = torch.ones(4, 8) * 10
val = triplet_loss(a, p, n).item()
assert val < 0.01, f"期望 loss < 0.01，实际 {val:.4f}"
print(f"✅ 检查1通过：loss = {val:.6f}（应接近0）")

# 检查2：anchor==negative, positive 很远 → loss ≈ d_pos + margin
a2 = torch.zeros(4, 8)
p2 = torch.ones(4, 8) * 10
n2 = torch.zeros(4, 8)
val2 = triplet_loss(a2, p2, n2).item()
d_pos_expected = (8 ** 0.5) * 10   # ||ones*10|| for dim=8
expected = d_pos_expected + 0.2
assert abs(val2 - expected) < 0.01, f"期望 {expected:.4f}，实际 {val2:.4f}"
print(f"✅ 检查2通过：loss = {val2:.4f}（应 ≈ {expected:.4f}）")

## 5. 参数实验：模拟三类音乐的 Embedding 聚类

**设置**：用随机生成的 mel 特征模拟 3 种风格（摇滚 / 爵士 / 古典），各 20 首歌，训练 50 轮。

**关键参数**：
- `margin=0.2`：过小 → 负样本靠得太近；过大 → 梯度消失（hard negative 不够）
- `embed_dim=128`：降到 16 可看到更明显聚类（维数诅咒减弱）
- `lr=1e-3`：收敛曲线若抖动，调低至 1e-4

**预期现象**：训练结束后 t-SNE 图中三类点应形成三簇，簇间距明显大于簇内距。

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ---------- 模拟数据 ----------
torch.manual_seed(42)
np.random.seed(42)

n_mels, T, embed_dim = 64, 50, 128
n_classes, n_per_class = 3, 20
# 每类中心不同，以使 CNN 能学到差异
class_centers = [
    torch.randn(1, 1, T, n_mels) * 2 + i * 3
    for i in range(n_classes)
]

def make_mel(class_id):
    """生成属于 class_id 的 mel 特征（加小噪声）。"""
    return class_centers[class_id] + torch.randn(1, 1, T, n_mels) * 0.3

all_mels = []  # (class_id, mel_tensor)
for c in range(n_classes):
    for _ in range(n_per_class):
        all_mels.append((c, make_mel(c)))

# ---------- 模型 & 优化器 ----------
encoder = MusicEncoder(n_mels=n_mels, embed_dim=embed_dim)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

# ---------- 训练 50 轮 ----------
losses = []
for epoch in range(50):
    epoch_loss = 0.0
    indices = torch.randperm(len(all_mels))
    for idx in indices:
        c_a, mel_a = all_mels[idx]
        # positive：同类随机另一首
        pos_idx = torch.randint(n_per_class, (1,)).item()
        _, mel_p = all_mels[c_a * n_per_class + pos_idx]
        # negative：不同类随机一首
        neg_class = (c_a + torch.randint(1, n_classes, (1,)).item()) % n_classes
        neg_idx = torch.randint(n_per_class, (1,)).item()
        _, mel_n = all_mels[neg_class * n_per_class + neg_idx]

        a = l2_normalize(encoder(mel_a))
        p = l2_normalize(encoder(mel_p))
        n_ = l2_normalize(encoder(mel_n))

        loss = triplet_loss(a, p, n_, margin=0.2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    losses.append(epoch_loss / len(all_mels))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/50  loss={losses[-1]:.4f}")

# ---------- t-SNE 可视化 ----------
encoder.eval()
embeds, labels = [], []
with torch.no_grad():
    for c, mel in all_mels:
        e = l2_normalize(encoder(mel)).squeeze(0).numpy()
        embeds.append(e)
        labels.append(c)
embeds = np.array(embeds)

try:
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt

    tsne = TSNE(n_components=2, perplexity=10, random_state=42)
    proj = tsne.fit_transform(embeds)

    colors = ['#e74c3c', '#3498db', '#2ecc71']
    names  = ['摇滚', '爵士', '古典']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # t-SNE 聚类图
    for c in range(n_classes):
        mask = np.array(labels) == c
        axes[0].scatter(proj[mask, 0], proj[mask, 1],
                        c=colors[c], label=names[c], s=60, alpha=0.8)
    axes[0].set_title("t-SNE：三类音乐 Embedding 聚类")
    axes[0].legend()

    # 训练 loss 曲线
    axes[1].plot(losses, color='#8e44ad')
    axes[1].set_title("训练 Loss 曲线（50 轮）")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Triplet Loss")

    plt.tight_layout()
    plt.show()
    print("✅ 可视化完成")
except ImportError:
    print("sklearn / matplotlib 未安装，跳过可视化")
    print(f"最终 loss: {losses[-1]:.4f}")

## 小结

本节实现了 `triplet_loss`，以 L2 距离驱动同歌向量靠近、异歌向量拉开，并通过 `MusicEncoder`（CNN + AdaptiveAvgPool）把可变长 mel 特征压缩为固定 128 维向量。输出经 `l2_normalize` 落在单位球面，使余弦相似度退化为点积，为后续 FAISS 检索节省计算开销。这套编码器构成 `aurora.music` 的语义骨干——下一节 **M4-MU3** 将基于此向量构建最近邻检索索引，实现毫秒级"找相似歌曲"。